In [2]:
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

# secret shhhhh
def get_secret():

    secret_name = "gitlab/tcwyu-respository"
    region_name = "us-east-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return secret
    
secret = ast.literal_eval(get_secret())


NoCredentialsError: Unable to locate credentials

In [ ]:
# git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/bertha_experiments.git',
    'username': secret['username'],
    'password': secret['password']
}

hyperparameters = {
    'epochs': 5, 
    'batch_size': 80, 
    'lr': 2.5e-5
}

estimator_args = {
    "role":"arn:aws:iam::776029861160:role/service-role/AmazonSageMaker-ExecutionRole-20230328T101238",
    "output_path":"s3://himss2023mimiciv/uw_skynet/output",
    "image_uri":"776029861160.dkr.ecr.us-east-2.amazonaws.com/smile-uw-containers:medair-huggingface-pytorch-training-test",
    "instance_count":1,
    "instance_type":"ml.m5.large",
    "entry_point":"AWS_load_data.py",
    "git_config":git_config,
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [ ]:
input_data = {
    "train": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/train",
    "test": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/test",
    "tokenizer": "s3://himss2023mimiciv/uw_skynet/data/tokenizer-mimic-iv-icd-final"
}

In [ ]:
pytorch_estimator.fit(inputs=input_data)

Cloning into '/tmp/tmp0w_byw2i'...
INFO:sagemaker:Creating training-job with name: smile-uw-containers-2023-03-31-19-06-50-334


2023-03-31 19:06:51 Starting - Starting the training job...
2023-03-31 19:07:05 Starting - Preparing the instances for training...
2023-03-31 19:07:45 Downloading - Downloading input data...
2023-03-31 19:08:10 Training - Downloading the training image..............................
2023-03-31 19:13:27 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
sed: can't read changehostname.c: No such file or directory
gcc: error: changehostname.c: No such file or directory
gcc: fatal error: no input files
compilation terminated.
gcc: error: changehostname.o: No such file or directory
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
2023-03-31 19:13:29,651 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2023-03-31 19:13:29,654 sagemaker-training-toolk

UnexpectedStatusException: Error for Training job smile-uw-containers-2023-03-31-19-06-50-334: Failed. Reason: AlgorithmError: ExecuteUserScriptError:
ExitCode 1
ErrorMessage ""
Command "/opt/conda/bin/python3.9 AWS_load_data.py", exit code: 1